# 04 · Stack

> Una lista Python usata solo dalla fine (LIFO): ti serve quando l'ultimo elemento aperto è il primo da chiudere (parentesi, espressioni) o quando cerchi il "prossimo maggiore/minore" (stack monotono).
> Idee per i singoli problemi: [Pattern algoritmici › Stack e stack monotono](../Fundamentals/Pattern%20algoritmici.md#stack-e-stack-monotono)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Valid Parentheses | ogni chiusa deve corrispondere all'ultima aperta | `dict` chiusa → aperta, `stack[-1]`, `not stack` |
| Min Stack | salva con ogni elemento il minimo fino a quel punto | `class`, `__init__`, stack di tuple `(val, min)` |
| Evaluate Reverse Polish Notation | operandi sullo stack, operatore = pop due, push risultato | `int(a / b)` (tronca verso 0), `dict` di `lambda` |
| Generate Parentheses | backtracking con contatori aperte/chiuse | ricorsione, `"".join(path)` → [Backtracking](10%20-%20Backtracking.ipynb) |
| Daily Temperatures | stack monotono decrescente di indici | `while stack and t > temps[stack[-1]]` |
| Car Fleet | ordina per posizione, calcola tempi di arrivo | `sorted(zip(pos, speed), reverse=True)`, divisione `/` |
| Largest Rectangle in Histogram | stack monotono crescente; quando estrai, calcoli l'area | tuple `(indice_inizio, altezza)`, passata finale |

## Sintassi al volo

| Operazione | Sintassi | Costo |
|---|---|---|
| push | `stack.append(x)` | `O(1)` |
| pop | `stack.pop()` (errore se vuoto) | `O(1)` |
| top | `stack[-1]` | `O(1)` |
| vuoto? | `if not stack:` / `while stack:` | `O(1)` |
| dimensione | `len(stack)` | `O(1)` |

## Corrispondenza aperta/chiusa

Mappa ogni simbolo di chiusura al suo di apertura. Alla chiusura controlli che lo stack non sia vuoto **prima** di leggere `stack[-1]`; alla fine lo stack deve essere vuoto.

In [1]:
pairs = {")": "(", "]": "[", "}": "{"}

stack = []
for ch in "{[(":
    stack.append(ch)
print(stack, stack[-1])  # -> ['{', '[', '('] (

closing = ")"
print(bool(stack) and stack[-1] == pairs[closing])  # -> True
stack.pop()
print(stack, len(stack) == 0)  # -> ['{', '['] False

['{', '[', '('] (
True
['{', '['] False


## Classi per i problemi "design"

LeetCode chiede spesso di implementare una classe. Lo stato vive negli attributi creati in `__init__`; ogni metodo riceve `self`.

Il trucco per *Min Stack*: invece di cercare il minimo, salva **con ogni elemento** il minimo valido in quel momento.

In [2]:
class PriceHistory:
    """Stack di prezzi che ricorda anche il massimo raggiunto a ogni livello."""

    def __init__(self) -> None:
        self.items: list[tuple[int, int]] = []  # (prezzo, massimo fino a qui)

    def push(self, price: int) -> None:
        current_max = max(price, self.items[-1][1]) if self.items else price
        self.items.append((price, current_max))

    def pop(self) -> int:
        return self.items.pop()[0]

    def max(self) -> int:
        return self.items[-1][1]

history = PriceHistory()
for p in (1200, 2599, 990):
    history.push(p)
print(history.max())  # -> 2599
history.pop(); history.pop()
print(history.max())  # -> 1200

2599
1200


## Operatori in un dict e divisione intera

Un dict `simbolo → funzione` evita una catena di `if`. Attenzione alla divisione: `//` arrotonda verso **-∞**, mentre molti problemi (e C/Java) troncano verso **zero**.

In [3]:
import operator

ops = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": lambda a, b: int(a / b),  # tronca verso zero
}
print(ops["-"](7, 2), ops["/"](-7, 2))  # -> 5 -3
print(-7 // 2, int(-7 / 2))             # -> -4 -3

stack = [10, 4]
b = stack.pop()   # il secondo operando esce per PRIMO
a = stack.pop()
print(ops["-"](a, b))  # -> 6

print("-11".isdigit(), int("-11"))  # -> False -11: per riconoscere i numeri controlla se è un operatore

5 -3
-4 -3
6
False -11


## Stack monotono: "prossimo elemento maggiore"

Tieni nello stack gli **indici** che aspettano ancora la risposta, con valori decrescenti. Quando arriva un valore più grande, risolve tutti quelli più piccoli in cima. Ogni indice entra ed esce una volta: `O(n)`.

In [4]:
daily_sales = [30, 20, 25, 40, 10]
days_until_higher = [0] * len(daily_sales)

stack: list[int] = []  # indici in attesa, valori decrescenti
for i, sales in enumerate(daily_sales):
    while stack and sales > daily_sales[stack[-1]]:
        j = stack.pop()
        days_until_higher[j] = i - j
    stack.append(i)
print(days_until_higher)  # -> [3, 1, 1, 0, 0]
print(stack)              # -> [3, 4]: nessun valore maggiore dopo di loro

[3, 1, 1, 0, 0]
[3, 4]


**Variante crescente (istogrammi):** con lo stack crescente, quando estrai una barra sai che non può estendersi oltre `i` a destra. Se salvi `(indice_inizio, altezza)`, la barra corrente "eredita" l'indice di inizio dell'ultima estratta, perché può estendersi fin lì a sinistra. Alla fine, gli elementi rimasti nello stack arrivano fino a `len(heights)`.

In [5]:
heights = [2, 4, 3]
stack: list[tuple[int, int]] = []  # (indice di inizio, altezza)
for i, h in enumerate(heights):
    start = i
    while stack and stack[-1][1] > h:
        start, popped_h = stack.pop()
        print("estratta", popped_h, "larghezza", i - start)  # -> estratta 4 larghezza 1
    stack.append((start, h))
print(stack)  # -> [(0, 2), (1, 3)]: l'altezza 3 parte dall'indice 1

estratta 4 larghezza 1
[(0, 2), (1, 3)]


## Ordinare coppie e confrontare tempi

`zip` accoppia due liste; ordinare le tuple ordina per il primo campo (poi il secondo). Per tempi e velocità usa la divisione vera `/`, che restituisce un `float`.

In [6]:
positions = [10, 8, 0, 5]
speeds = [2, 4, 1, 1]
target = 12

for pos, speed in sorted(zip(positions, speeds), reverse=True):  # dal più vicino al traguardo
    time = (target - pos) / speed
    print(pos, time)  # -> 10 1.0 / 8 1.0 / 5 7.0 / 0 12.0

10 1.0
8 1.0
5 7.0
0 12.0


## Errori comuni

- `stack[-1]` o `stack.pop()` su stack vuoto: `IndexError`. Controlla `if stack` prima.
- Salvare **valori** invece di indici nello stack monotono quando ti serve una distanza.
- RPN: l'ordine degli operandi conta per `-` e `/` (il primo estratto è il secondo operando).
- Dimenticare gli elementi rimasti nello stack alla fine (istogramma).